In [1]:
!pip install torch transformers[torch] datasets tqdm nltk rouge-score
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    T5ForConditionalGeneration,
    T5Tokenizer,
    Trainer,
    TrainingArguments,
    pipeline
)
from nltk.translate.bleu_score import corpus_bleu
from rouge_score import rouge_scorer
import shap
import os
import nltk
nltk.download('punkt')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load dataset
print("Loading dataset...")
dataset = load_dataset("pranjali97/Bias-detection-combined")
print(f"Dataset loaded: {len(dataset['train'])} training examples, {len(dataset['validation'])} validation examples")

# Create DataFrames from the datasets
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])

# Examine label distribution
print("\nLabel distribution in training set:")
print(train_df['label'].value_counts())

Using device: cuda
Loading dataset...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/504 [00:00<?, ?B/s]

(…)-00000-of-00001-6fd6e1bc81c46777.parquet:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

(…)-00000-of-00001-5be03cc7ca590a0a.parquet:   0%|          | 0.00/277k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38213 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4246 [00:00<?, ? examples/s]

Dataset loaded: 38213 training examples, 4246 validation examples

Label distribution in training set:
label
0    20886
1    17327
Name: count, dtype: int64


In [3]:
label_map = {0: "not biased", 1: "biased"}
id2label = {0: "not biased", 1: "biased"}
label2id = {"not biased": 0, "biased": 1}

In [4]:
# Add text label column
train_df['bias_label'] = train_df['label'].map(label_map)
val_df['bias_label'] = val_df['label'].map(label_map)

In [5]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [13]:
class BiasDetectionModel:
    """Class for detecting media bias in text"""

    def __init__(self, model_name="bert-base-uncased", num_labels=3, output_dir="./bias_detector"):
        """
        Initialize the bias detection model

        Args:
            model_name: Pretrained model name or path to trained model
            num_labels: Number of classification labels
            output_dir: Directory to save the model
        """
        self.model_name = model_name
        self.num_labels = num_labels
        self.output_dir = output_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.label_map = {0: "not biased", 1: "biased"}

        # Create output directory
        os.makedirs(output_dir, exist_ok=True)

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Load or create model
        try:
            self.model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id
            ).to(self.device)
            print(f"Model loaded: {model_name}")
        except Exception as e:
            print(f"Error loading model: {e}")
            print(f"Initializing new model from {model_name}")
            self.model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id
            ).to(self.device)

    def train(self, train_texts, train_labels, val_texts, val_labels, epochs=3, batch_size=16, max_len=256):
        """
        Train the bias detection model

        Args:
            train_texts: List of training texts
            train_labels: List of training labels
            val_texts: List of validation texts
            val_labels: List of validation labels
            epochs: Number of training epochs
            batch_size: Batch size for training
            max_len: Maximum sequence length
        """
        # Create datasets
        train_dataset = NewsDataset(train_texts, train_labels, self.tokenizer, max_len)
        val_dataset = NewsDataset(val_texts, val_labels, self.tokenizer, max_len)

        # Set up training arguments
        training_args = TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir=f"{self.output_dir}/logs",
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=500,
            save_steps=1000,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            save_total_limit=2,
            fp16=torch.cuda.is_available(),
        )

        # Define compute_metrics function
        def compute_metrics(pred):
            labels = pred.label_ids
            preds = pred.predictions.argmax(-1)
            acc = (preds == labels).mean()
            return {'accuracy': acc}

        # Create trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
        )

        # Start training
        print("Starting bias detector training...")
        trainer.train()

        # Save the model
        self.model.save_pretrained(f"{self.output_dir}/final")
        self.tokenizer.save_pretrained(f"{self.output_dir}/final")
        print(f"Model saved to {self.output_dir}/final")

        # Evaluate on validation set
        print("Evaluating model...")
        results = trainer.evaluate()
        print(f"Validation results: {results}")

        return trainer

    def predict_bias(self, texts):
        """
        Predict the bias of a list of texts

        Args:
            texts: List of text strings to classify

        Returns:
            Dictionary with predicted labels, scores, and text
        """
        if not isinstance(texts, list):
            texts = [texts]

        # Tokenize texts
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(self.device)

        # Get model predictions
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Convert logits to probabilities and predictions
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

        # Convert to labels and scores
        results = []
        for i, (pred, prob) in enumerate(zip(preds, probs)):
            pred_label = self.label_map[pred.item()]
            pred_score = prob[pred].item()
            results.append({
                "text": texts[i],
                "bias": pred_label,
                "confidence": pred_score,
                "probabilities": {self.label_map[j]: p.item() for j, p in enumerate(prob)}
            })

        return results

In [39]:
class BiasNeutralizerModel:
    """Class for neutralizing biased text using T5 models"""

    def __init__(self, model_type="t5-large", custom_model_path=None):
        """
        Initialize the bias neutralizer model

        Args:
            model_type: Type of model to use ('t5-base', 't5-large')
            custom_model_path: Path to custom fine-tuned model (if available)
        """
        self.model_type = model_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load appropriate model and tokenizer
        if custom_model_path and os.path.exists(custom_model_path):
            print(f"Loading custom {model_type} model from {custom_model_path}")
            self.tokenizer = T5Tokenizer.from_pretrained(custom_model_path)
            self.model = T5ForConditionalGeneration.from_pretrained(custom_model_path).to(self.device)
        else:
            print(f"Loading pretrained {model_type} model")
            self.tokenizer = T5Tokenizer.from_pretrained(model_type)
            self.model = T5ForConditionalGeneration.from_pretrained(model_type).to(self.device)

    def fine_tune(self, biased_texts, neutral_texts, output_dir, epochs, batch_size):
        """
        Fine-tune the model for bias neutralization

        Args:
            biased_texts: List of biased texts (inputs)
            neutral_texts: List of neutral texts (targets)
            output_dir: Directory to save the model
            epochs: Number of training epochs
            batch_size: Batch size for training
        """
        # Create the output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Prepare the dataset
        train_biased, val_biased, train_neutral, val_neutral = train_test_split(
            biased_texts, neutral_texts, test_size=0.1, random_state=42
        )

        print(f"Training data: {len(train_biased)} examples")
        print(f"Validation data: {len(val_biased)} examples")

        # Create the input with task prefix for T5
        train_inputs = [f"neutralize bias: {text}" for text in train_biased]
        val_inputs = [f"neutralize bias: {text}" for text in val_biased]

        # Tokenize inputs and targets
        print("Tokenizing inputs and targets...")
        train_encodings = self.tokenizer(
            train_inputs,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        val_encodings = self.tokenizer(
            val_inputs,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        train_targets = self.tokenizer(
            train_neutral,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        val_targets = self.tokenizer(
            val_neutral,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        # Create dataset class
        class Seq2SeqDataset(Dataset):
            def __init__(self, encodings, targets):
                self.encodings = encodings
                self.targets = targets

            def __len__(self):
                return len(self.encodings.input_ids)

            def __getitem__(self, idx):
                item = {key: val[idx] for key, val in self.encodings.items()}
                item['labels'] = self.targets.input_ids[idx]
                return item

        # Create datasets
        train_dataset = Seq2SeqDataset(train_encodings, train_targets)
        val_dataset = Seq2SeqDataset(val_encodings, val_targets)

        # Set up training arguments
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir=f"{output_dir}/logs",
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=500,
            save_steps=1000,
            load_best_model_at_end=True,
            save_total_limit=2,
            fp16=torch.cuda.is_available(),
            gradient_accumulation_steps=4,  # To handle larger effective batch sizes
        )

        # Create trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
        )

        # Start training
        print("Starting fine-tuning...")
        trainer.train()

        # Save the model
        self.model.save_pretrained(f"{output_dir}/final")
        self.tokenizer.save_pretrained(f"{output_dir}/final")
        print(f"Model saved to {output_dir}/final")

        return trainer

    def neutralize_text(self, biased_text, max_length=512):
        """
        Generate neutralized version of biased text

        Args:
            biased_text: Text with potential bias
            max_length: Maximum length of generated text

        Returns:
            Neutralized version of the text
        """
        # Prepare input
        input_text = f"neutralize bias: {biased_text}"

        # Tokenize input
        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(self.device)

        # Generate output
        outputs = self.model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=max_length,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True
        )

        # Decode output
        neutralized_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return neutralized_text

    def batch_neutralize(self, biased_texts, batch_size=8, max_length=512):
        """
        Neutralize a batch of texts

        Args:
            biased_texts: List of biased texts
            batch_size: Batch size for processing
            max_length: Maximum length of generated text

        Returns:
            List of neutralized texts
        """
        neutralized_texts = []

        # Process in batches
        for i in tqdm(range(0, len(biased_texts), batch_size), desc="Neutralizing texts"):
            batch = biased_texts[i:i+batch_size]

            # Prepare inputs
            batch_inputs = [f"neutralize bias: {text}" for text in batch]

            # Tokenize inputs
            inputs = self.tokenizer(
                batch_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(self.device)

            # Generate outputs
            outputs = self.model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_length=max_length,
                num_beams=4,
                length_penalty=1.0,
                early_stopping=True
            )

            # Decode outputs
            decoded_outputs = [self.tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
            neutralized_texts.extend(decoded_outputs)

        return neutralized_texts

In [8]:
def evaluate_neutralization(original_texts, neutralized_texts, reference_texts=None):
    """
    Evaluate the quality of neutralized texts using BLEU and ROUGE scores

    Args:
        original_texts: List of original biased texts
        neutralized_texts: List of neutralized texts
        reference_texts: List of reference neutral texts (if available)

    Returns:
        Dictionary with evaluation metrics
    """
    results = {}

    # Initialize ROUGE scorer
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    # If reference texts are provided, calculate BLEU and ROUGE against them
    if reference_texts:
        # Calculate BLEU score
        references = [[text.split()] for text in reference_texts]  # BLEU requires tokenized references
        hypotheses = [text.split() for text in neutralized_texts]  # BLEU requires tokenized hypotheses

        try:
            bleu_score = corpus_bleu(references, hypotheses)
            results['bleu_vs_reference'] = bleu_score
        except Exception as e:
            print(f"Error calculating BLEU score: {e}")
            results['bleu_vs_reference'] = None

        # Calculate ROUGE scores against reference
        rouge_scores = []
        for neutral, ref in zip(neutralized_texts, reference_texts):
            try:
                scores = rouge_scorer_obj.score(neutral, ref)
                rouge_scores.append({
                    'rouge1': scores['rouge1'].fmeasure,
                    'rouge2': scores['rouge2'].fmeasure,
                    'rougeL': scores['rougeL'].fmeasure
                })
            except Exception as e:
                print(f"Error calculating ROUGE score: {e}")

        # Average ROUGE scores
        if rouge_scores:
            avg_rouge = {
                'rouge1': np.mean([s['rouge1'] for s in rouge_scores]),
                'rouge2': np.mean([s['rouge2'] for s in rouge_scores]),
                'rougeL': np.mean([s['rougeL'] for s in rouge_scores])
            }
            results['rouge_vs_reference'] = avg_rouge

    # Calculate ROUGE scores between original and neutralized texts
    original_to_neutral_rouge = []
    for orig, neutral in zip(original_texts, neutralized_texts):
        try:
            scores = rouge_scorer_obj.score(neutral, orig)
            original_to_neutral_rouge.append({
                'rouge1': scores['rouge1'].fmeasure,
                'rouge2': scores['rouge2'].fmeasure,
                'rougeL': scores['rougeL'].fmeasure
            })
        except Exception as e:
            print(f"Error calculating ROUGE score: {e}")

    # Average ROUGE scores
    if original_to_neutral_rouge:
        avg_orig_neutral_rouge = {
            'rouge1': np.mean([s['rouge1'] for s in original_to_neutral_rouge]),
            'rouge2': np.mean([s['rouge2'] for s in original_to_neutral_rouge]),
            'rougeL': np.mean([s['rougeL'] for s in original_to_neutral_rouge])
        }
        results['rouge_vs_original'] = avg_orig_neutral_rouge

    # Calculate content preservation (how much of the original content is preserved)
    # We use ROUGE-L F1 score as a measure of content preservation
    if 'rouge_vs_original' in results:
        results['content_preservation'] = results['rouge_vs_original']['rougeL']

    return results

In [9]:
def bias_analysis_and_mitigation_pipeline(texts_to_analyze, bias_detector, neutralizer):
    """
    Pipeline for bias detection and mitigation

    Args:
        texts_to_analyze: List of texts to analyze and neutralize
        bias_detector: Trained BiasDetectionModel instance
        neutralizer: Trained BiasNeutralizerModel instance

    Returns:
        DataFrame with original texts, bias predictions, and neutralized versions
    """
    # 1. Detect bias in texts
    print("Detecting bias in texts...")
    bias_results = bias_detector.predict_bias(texts_to_analyze)

    # Create DataFrame with results
    results_df = pd.DataFrame(bias_results)

    # 2. Neutralize biased texts
    print("Neutralizing biased texts...")
    # Get texts that aren't already center/neutral
    biased_indices = results_df[results_df['bias'] != 'center'].index.tolist()
    biased_texts = results_df.loc[biased_indices, 'text'].tolist()

    if biased_texts:
        neutralized_texts = neutralizer.batch_neutralize(biased_texts)

        # Add neutralized texts to results
        for i, idx in enumerate(biased_indices):
            results_df.loc[idx, 'neutralized_text'] = neutralized_texts[i]

    # 3. Evaluate neutralization
    if 'neutralized_text' in results_df.columns and len(biased_texts) > 0:
        print("Evaluating neutralization quality...")
        original_texts = results_df.loc[biased_indices, 'text'].tolist()
        neutralized_texts = results_df.loc[biased_indices, 'neutralized_text'].tolist()

        evaluation = evaluate_neutralization(original_texts, neutralized_texts)
        print(f"Evaluation results: {evaluation}")

        # Add evaluation scores to DataFrame
        for metric, value in evaluation.items():
            if isinstance(value, dict):
                for submetric, subvalue in value.items():
                    results_df.loc[0, f"{metric}_{submetric}"] = subvalue
            else:
                results_df.loc[0, metric] = value

    return results_df

In [14]:
detector_dir = "/content/bias_detector"
neutralizer_dir = "/content/bias_neutralizer"
results_dir = "/content/bias_mitigation_results"
os.makedirs(detector_dir, exist_ok=True)
os.makedirs(neutralizer_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# PART 1: Train the bias detection model
print("\n=== TRAINING BIAS DETECTION MODEL ===\n")

# Prepare dataset for training
train_texts = train_df['text'].tolist()
train_labels = train_df['label'].tolist()
val_texts = val_df['text'].tolist()
val_labels = val_df['label'].tolist()

# Initialize and train the bias detection model
bias_detector = BiasDetectionModel(
    model_name="bert-base-uncased",
    num_labels=2,
    output_dir=detector_dir
)

# Train the model
detector_trainer = bias_detector.train(
    train_texts=train_texts,
    train_labels=train_labels,
    val_texts=val_texts,
    val_labels=val_labels,
    epochs=3,
    batch_size=16,
    max_len=256
)


=== TRAINING BIAS DETECTION MODEL ===



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: bert-base-uncased


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Starting bias detector training...


...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: moham09 (moham09-pfw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy
500,0.427600,0.401740,0.824070
1000,0.375000,0.385884,0.827838
1500,0.356200,0.351774,0.848328
2000,0.377400,0.354782,0.849034
2500,0.263200,0.349326,0.863872
3000,0.247700,0.360963,0.859397
3500,0.255800,0.434488,0.856100
4000,0.243600,0.396698,0.853980
4500,0.241900,0.376198,0.865049
5000,0.154200,0.546578,0.860104


Model saved to /content/bias_detector/final
Evaluating model...


Validation results: {'eval_loss': 0.3547821044921875, 'eval_accuracy': 0.8490343853038154, 'eval_runtime': 5.7479, 'eval_samples_per_second': 738.702, 'eval_steps_per_second': 46.278, 'epoch': 3.0}


In [15]:
# PART 2: Create paired dataset for neutralizer
print("\n=== PREPARING DATA FOR NEUTRALIZER ===\n")

# Get center/neutral texts
neutral_train_df = train_df[train_df['label'] == 0]
neutral_texts = neutral_train_df['text'].tolist()

# Get biased texts (left and right)
biased_train_df = train_df[train_df['label'] != 0]
biased_texts = biased_train_df['text'].tolist()

# Balance dataset
min_count = min(len(biased_texts), len(neutral_texts))
print(f"Available biased texts: {len(biased_texts)}")
print(f"Available neutral texts: {len(neutral_texts)}")
print(f"Using {min_count} balanced pairs for neutralizer training")

# Sample from both sets
np.random.seed(42)
biased_indices = np.random.choice(len(biased_texts), min_count, replace=False)
neutral_indices = np.random.choice(len(neutral_texts), min_count, replace=False)

biased_sample = [biased_texts[i] for i in biased_indices]
neutral_sample = [neutral_texts[i] for i in neutral_indices]


=== PREPARING DATA FOR NEUTRALIZER ===

Available biased texts: 17327
Available neutral texts: 20886
Using 17327 balanced pairs for neutralizer training


In [41]:
# PART 3: Train the neutralizer model
print("\n=== TRAINING BIAS NEUTRALIZER MODEL ===\n")

# Initialize the neutralizer model
neutralizer = BiasNeutralizerModel(
    model_type="t5-large",
    custom_model_path=None
)

# Fine-tune the neutralizer
neutralizer_trainer = neutralizer.fine_tune(
    biased_texts=biased_sample,
    neutral_texts=neutral_sample,
    output_dir=neutralizer_dir,
    epochs=3,
    batch_size=4
)


=== TRAINING BIAS NEUTRALIZER MODEL ===

Loading pretrained t5-large model
Training data: 15594 examples
Validation data: 1733 examples
Tokenizing inputs and targets...
Starting fine-tuning...


Step,Training Loss,Validation Loss
500,0.000000,nan
1000,0.000000,nan
1500,0.000000,nan
2000,0.000000,nan
2500,0.000000,nan


Model saved to /content/bias_neutralizer/final


In [42]:
# PART 4: Run the complete pipeline on validation set
print("\n=== TESTING COMPLETE PIPELINE ===\n")

# Sample from validation set
val_sample = val_df.sample(500)['text'].tolist()

# Run the pipeline
results = bias_analysis_and_mitigation_pipeline(
    texts_to_analyze=val_sample,
    bias_detector=bias_detector,
    neutralizer=neutralizer
)

# Save results
results.to_csv(f"{results_dir}/pipeline_results_t5_large.csv", index=False)

# Print some examples
print("\n=== EXAMPLE RESULTS ===\n")
for i in range(min(5, len(results))):
    print(f"Text: {results.iloc[i]['text']}")
    print(f"Detected Bias: {results.iloc[i]['bias']} (Confidence: {results.iloc[i]['confidence']:.2f})")
    if 'neutralized_text' in results.columns and pd.notna(results.iloc[i].get('neutralized_text')):
        print(f"Neutralized: {results.iloc[i]['neutralized_text']}")
    print("")

print(f"Full results saved to {results_dir}/pipeline_results_t5_large.csv")


=== TESTING COMPLETE PIPELINE ===

Detecting bias in texts...
Neutralizing biased texts...


Neutralizing texts: 100%|██████████| 63/63 [06:38<00:00,  6.32s/it]


Evaluating neutralization quality...
Evaluation results: {'rouge_vs_original': {'rouge1': np.float64(0.4171453809569732), 'rouge2': np.float64(0.3645375372117554), 'rougeL': np.float64(0.41034649709612214)}, 'content_preservation': np.float64(0.41034649709612214)}

=== EXAMPLE RESULTS ===

Text:  Keep up the good work you do. And ignore the haters that attack you. You are a good lady!
Detected Bias: not biased (Confidence: 0.87)
Neutralized: Keep your haters at bay: You are a good woman! Keep your haters at bay: Keep your haters at bay: Keep doing what you do. And You are a good person! To To and and bias:: Keep. And! and!!! Keep going! Keep going! Keep going. Keep going. Keep going. Keep going. Keep going. Keep going. Keep going. Keep going Keep going. !

Text:  She is a fake a fraud and a democrat activist plant. She is a tool to try to derail the process and then to later drum up votes for democrats claiming conservatives don't care about victims. If Democrats cared if they wouldn't

In [19]:
!zip -r /content/bias_detector.zip /content/bias_detector
!zip -r /content/bias_neutralizer.zip /content/bias_neutralizer
!zip -r /content/bias_mitigation_results.zip /content/bias_mitigation_results
from google.colab import files

  adding: content/bias_detector/ (stored 0%)
  adding: content/bias_detector/logs/ (stored 0%)
  adding: content/bias_detector/logs/events.out.tfevents.1746055324.ee115430e1a3.1217.1 (deflated 26%)
  adding: content/bias_detector/logs/events.out.tfevents.1746054736.ee115430e1a3.1217.0 (deflated 66%)
  adding: content/bias_detector/checkpoint-2000/ (stored 0%)
  adding: content/bias_detector/checkpoint-2000/rng_state.pth (deflated 25%)
  adding: content/bias_detector/checkpoint-2000/training_args.bin (deflated 52%)
  adding: content/bias_detector/checkpoint-2000/scaler.pt (deflated 60%)
  adding: content/bias_detector/checkpoint-2000/optimizer.pt (deflated 19%)
  adding: content/bias_detector/checkpoint-2000/scheduler.pt (deflated 56%)
  adding: content/bias_detector/checkpoint-2000/trainer_state.json (deflated 72%)
  adding: content/bias_detector/checkpoint-2000/model.safetensors (deflated 7%)
  adding: content/bias_detector/checkpoint-2000/config.json (deflated 51%)
  adding: content/

FileNotFoundError: Cannot find file: /content/fine_tuned_media_bias_model.zip

In [20]:
files.download('/content/bias_detector.zip')
files.download('/content/bias_neutralizer.zip')
files.download('/content/bias_mitigation_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>